# 03 · Gold activity

Combine saved Silver data into one row per vehicle and UTC day. This is the data source for a separate Power BI report.

Run after `02_silver_cleaning`. Gold reads tables, not variables from Silver. Null distance means missing or unreliable data; it is not zero.

## Paths, dates and imports

In [ ]:
from pathlib import Path
from datetime import date, datetime, timezone
import json
import os
import sqlite3
import uuid

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema_name = "ev_telematics.fleet_activity_demo_20260910"
project_dir = Path("/Volumes/ev_telematics/fleet_activity_demo_20260910/files")
source_dir = project_dir / "data" / "sources"
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)

# This notebook uses the Databricks-provided Spark session.
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Report period:", start_date, "to", end_date)

## Read the saved Silver tables

In [ ]:
daily = spark.table(schema_name + ".daily_observations")
fleet = spark.table(schema_name + ".silver_fleet")
weather = spark.table(schema_name + ".silver_weather")

## Build the reporting dates
One date per day in the reporting period. A simple loop builds the small demo calendar.

In [ ]:
from datetime import timedelta

calendar_rows = []
current_day = start_date
while current_day <= end_date:
    # The trailing comma makes a one-column row.
    calendar_rows.append((current_day,))
    current_day = current_day + timedelta(days=1)
report_dates = spark.createDataFrame(calendar_rows, "event_date date")
print("Reporting days:", report_dates.count())

## Keep missing and unknown vehicle-days
`crossJoin` creates the expected vehicle × day combinations. A left join keeps missing telemetry. The anti-join finds vehicles absent from the roster. Expected: 720 roster-days + one unknown vehicle-day.

In [ ]:
expected_vehicle_days = fleet.select("vehicle_id").crossJoin(report_dates)
registered_days = expected_vehicle_days.join(daily, ["vehicle_id", "event_date"], "left")
unregistered_days = daily.join(fleet.select("vehicle_id"), "vehicle_id", "left_anti")
vehicle_days = registered_days.unionByName(unregistered_days)
vehicle_days = (vehicle_days.join(fleet, "vehicle_id", "left")
    .withColumn("registered_vehicle", F.col("depot_id").isNotNull()))
vehicle_days = (vehicle_days
    .withColumn("distance_status", F.coalesce("distance_status", F.lit("missing_telemetry")))
    .withColumn("reading_count", F.coalesce("reading_count", F.lit(0)))
    .withColumn("depot_id", F.coalesce("depot_id", F.lit("UNASSIGNED")))
    .withColumn("depot_name", F.coalesce("depot_name", F.lit("Unknown depot")))
    .withColumn("model", F.coalesce("model", F.lit("Unknown model"))))
vehicle_days.groupBy("distance_status").count().show()

## Join weather and publish Gold
Join on **depot and date**. A weather row must match many vehicles without multiplying their rows. Precipitation ≥ 1 mm is Wet; a successful null observation is Unknown.

In [ ]:
gold = (vehicle_days.join(weather, ["depot_id", "event_date"], "left")
    .withColumn("weather_condition", F.when(F.col("precipitation_mm").isNull(), "Unknown")
        .when(F.col("precipitation_mm") >= 1.0, "Wet").otherwise("Dry"))
    .select("event_date", "vehicle_id", "depot_id", "depot_name", "model", "registered_vehicle",
        "reading_count", "first_reading", "last_reading", "distance_status", "observed_distance_km",
        "temperature_c", "precipitation_mm", "weather_condition"))
if gold.count() != vehicle_days.count():
    raise ValueError("Weather join multiplied or dropped vehicle-days")
if gold.select("event_date", "vehicle_id").distinct().count() != gold.count():
    raise ValueError("Gold grain is not unique")
non_trusted_distance = gold.filter(
    (F.col("distance_status") != "trusted") & F.col("observed_distance_km").isNotNull())
if non_trusted_distance.count() != 0:
    raise ValueError("Non-trusted day has a distance")
if gold.filter(F.col("observed_distance_km") < 0).count() != 0:
    raise ValueError("Negative observed distance")

gold_table = schema_name + ".gold_daily_vehicle_activity"
gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table)
gold = spark.table(gold_table)
gold.createOrReplaceTempView("daily_vehicle_activity")

## Check Gold with SQL
This small summary is a validation reference for the separate Power BI report. Keep coverage beside averages so missing or unreliable days are not mistaken for zero activity.

In [ ]:
activity_report = spark.sql("""
    SELECT depot_name, model, weather_condition,
           COUNT(*) AS vehicle_days,
           COUNT(observed_distance_km) AS trusted_vehicle_days,
           SUM(CASE WHEN distance_status = 'missing_telemetry' THEN 1 ELSE 0 END) AS missing_days,
           SUM(CASE WHEN distance_status = 'untrusted' THEN 1 ELSE 0 END) AS untrusted_days,
           SUM(CASE WHEN distance_status = 'insufficient_readings' THEN 1 ELSE 0 END) AS insufficient_days,
           ROUND(AVG(observed_distance_km), 2) AS mean_observed_distance_km,
           ROUND(SUM(observed_distance_km), 2) AS total_observed_distance_km
    FROM daily_vehicle_activity
    GROUP BY depot_name, model, weather_condition
    ORDER BY depot_name, model, weather_condition
""")
activity_report.show(truncate=False)

## Export Gold for Power BI

Save the Gold data as CSV for the separate Power BI report.


In [ ]:
# These are small demo outputs; collecting is deliberate, not a large-data pattern.
import csv
# Format timestamps in Spark's UTC session before Python collects them.
daily_export = (gold.orderBy("event_date", "vehicle_id")
    .withColumn("first_reading", F.date_format("first_reading", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("last_reading", F.date_format("last_reading", "yyyy-MM-dd'T'HH:mm:ss'Z'")))
for name, frame in [("daily_vehicle_activity", daily_export),
                    ("activity_report", activity_report)]:
    with (output_dir / f"{name}.csv").open("w", newline="") as file:
        writer = csv.writer(file, lineterminator="\n")
        writer.writerow(frame.columns)
        writer.writerows(tuple(row) for row in frame.collect())

## Appendix: execution evidence
Read the saved tables to record counts and file hashes. Duplicate count follows the reconciliation: Bronze = accepted + quarantine + removed duplicates. This is supporting evidence, not the Power BI report.

In [ ]:
bronze = spark.table(schema_name + ".bronze_telemetry")
quarantine = spark.table(schema_name + ".quarantine_telemetry")
accepted = spark.table(schema_name + ".silver_telemetry")
day_issues = spark.table(schema_name + ".silver_day_issues")
row_quarantine = quarantine.filter(F.col("quarantine_scope") == "record")
day_quarantine = quarantine.filter(F.col("quarantine_scope") == "day_hold")
exact_duplicates_removed = bronze.count() - accepted.count() - quarantine.count()
run_id = bronze.select("run_id").first()["run_id"]
database_path = source_dir / "fleet.sqlite"
weather_dir = project_dir / "examples/weather"
weather_snapshots = []
for path in sorted(weather_dir.glob("*.json")):
    snapshot = json.loads(path.read_text())
    weather_snapshots.append({"file": path.name, "request": snapshot["request"], "fetched_at": snapshot["fetched_at"]})

import hashlib
unassigned_rejects = quarantine.filter(
    F.col("vehicle_id").isNull() | (F.col("vehicle_id") == "") | F.col("event_date").isNull()).count()
record_reason_counts = {row.reason: row["count"] for row in
    row_quarantine.select(F.explode("reasons").alias("reason")).groupBy("reason").count().collect()}
day_issue_counts = {row.reason: row["count"] for row in
    day_issues.select(F.explode("reasons").alias("reason")).groupBy("reason").count().collect()}
status_counts = {row.distance_status: row["count"] for row in gold.groupBy("distance_status").count().collect()}
input_files = sorted((source_dir / "telemetry").glob("*.jsonl")) + [database_path]
input_files += sorted(weather_dir.glob("*.json"))
summary = {
    "run_id": run_id, "completed_at": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version, "period": [str(start_date), str(end_date)],
    "bronze_rows": bronze.count(), "exact_duplicates_removed": exact_duplicates_removed,
    "quarantined_rows": quarantine.count(), "unassigned_rejects": unassigned_rejects,
    "individual_reject_rows": row_quarantine.count(), "held_rows": day_quarantine.count(),
    "individual_reject_reason_counts": record_reason_counts, "untrusted_day_reason_counts": day_issue_counts,
    "gold_rows": gold.count(), "status_counts": status_counts,
    "unregistered_vehicle_days": gold.filter(~F.col("registered_vehicle")).count(),
    "weather_rows": weather.count(), "weather_snapshots": weather_snapshots,
    "inputs": {str(path.relative_to(project_dir)): hashlib.sha256(path.read_bytes()).hexdigest() for path in input_files},
    "business_output_sha256": hashlib.sha256((output_dir / "daily_vehicle_activity.csv").read_bytes()).hexdigest(),
}
(output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps({key: value for key, value in summary.items() if key not in ["inputs", "weather_snapshots"]}, indent=2))

## Verify against the saved demo reference

In [ ]:
# Compare the platform result with the verified local business output.
expected_hash = '766345118556c0b8901c8fbed8998a143c7741910c9faac6daade370e532c44c'
if summary["business_output_sha256"] != expected_hash:
    raise ValueError("Databricks business CSV differs from the local reference")
if summary["gold_rows"] != 721 or summary["quarantined_rows"] != 22:
    raise ValueError("Databricks counts differ from the local reference")
print("PLATFORM_REPLAY_VALIDATION_PASSED")
print("Weather mode: saved real API snapshots; no cloud fetch claimed")
print("Gold table:", gold_table)
print("Business SHA-256:", summary["business_output_sha256"])
print("Gold versions:")
spark.sql(f"DESCRIBE HISTORY {gold_table}").select("version", "timestamp", "operation").show()
display(gold.limit(10))

## Pipeline complete
Use `gold_daily_vehicle_activity` in Power BI. Build visuals there. No HTML, CSS or Matplotlib is required. See `POWER_BI.md` for the Gold fields and metric limits.